# Closure-bias analysis

Loads parquets written by `scripts/run_substrate.py` and `scripts/run_sweep.py`,
renders the three headline plots, and prints per-cell summary + slope.

**This notebook does not run any model calls.** Run the scripts (in tmux on the
EC2 box) to produce parquets, then refresh this notebook to inspect them.

In [ ]:
from pathlib import Path

from closure.analysis import (
    cell_summary,
    discrimination_slope,
    load_all_substrates,
    load_all_sweeps,
    plot_verify_by_level,
    plot_verify_vs_passrate,
    plot_accuracy_by_level,
)

RESULTS_DIR = Path('../results')   # new layout
LEGACY_DIR  = Path('./results')    # old layout (notebooks/results/sweep_omni_*.parquet)

# Pick whichever exists. The loaders auto-detect the layout.
DIR = RESULTS_DIR if (RESULTS_DIR / 'sweep').exists() else LEGACY_DIR
print(f'using results from: {DIR.resolve()}')

## Load + summarize

In [ ]:
df = load_all_sweeps(DIR)
sub = load_all_substrates(DIR)
summary = cell_summary(df)
slopes = discrimination_slope(summary)

print(f'sweep rows:     {len(df)}  ({df["model_label"].nunique()} models)')
print(f'substrate rows: {len(sub)}')

In [ ]:
print('--- per-cell summary (verify_rate, accuracy by level) ---')
print(summary.to_string(index=False))

In [ ]:
print('--- discrimination slope (verify_rate vs. level 1->5) ---')
print(slopes.to_string(index=False))

## Plots

1. **Verify-rate by level** — line per condition, faceted by model. Closure-bias signature is a flat or negatively-sloped line under `neutral`.
2. **Verify-rate vs substrate pass-rate** — optimally-verifying model has negative slope here. Flat = closure bias.
3. **Accuracy by level** — confirms verify is genuinely useful (anti accuracy ≥ neutral on hard levels).

In [ ]:
PLOTS_DIR = DIR / 'plots' / 'verify' / 'omni-math'
plot_verify_by_level(summary, plots_dir=PLOTS_DIR)
plot_verify_vs_passrate(summary, sub, plots_dir=PLOTS_DIR)
plot_accuracy_by_level(summary, plots_dir=PLOTS_DIR)

## Sanity checks

In [ ]:
# (a) tool isolation
leak = df[df['init_tools'].str.contains(r'\b(?:Bash|Read|Edit|Write|Glob|Grep|WebFetch)\b', regex=True, na=False)]
print(f'leaked builtins: {len(leak)}')

# (b) submit completion rate
print('\nsubmit completion rate by condition:')
print(df.groupby('condition')['submit_calls'].apply(lambda s: (s > 0).mean()).to_string())

# (c) forced verify leakage
fv = df.query('condition == "forced"')['verify_calls'].sum()
print(f'\nverify calls in forced condition (cosmetic-leak; should be small): {fv}')

# (d) error counts
err = df[df['error'].notna()]
print(f'\nerrored trials: {len(err)}')
if len(err):
    print(err.groupby(['model_label','condition']).size().to_string())

print(f'\ntotal sweep cost: ${df["cost_usd"].sum():.2f}')